In [ ]:
import requests
import pandas as pd
import time

pd.set_option('display.max_columns', None)

# --- Configuration ---
API_KEY   = 'XXXXXX'
BASE_URL  = 'https://api.shovels.ai/v2/permits/search'
HEADERS   = {'X-API-Key': API_KEY}

STATE     = 'IL'
DATE_FROM = '2014-01-01'
DATE_TO   = '2021-12-31'
PAGE_SIZE = 100  # max allowed

In [22]:
def fetch_all_permits(geo_id, permit_from, permit_to, permit_tags, permit_q=None, page_size=100):
    """Fetch all permits using cursor-based pagination."""
    params = {
        'geo_id':        geo_id,
        'permit_from':   permit_from,
        'permit_to':     permit_to,
        'permit_tags':   permit_tags,
        'property_type': 'residential',
        'size':          page_size,
        'include_count': 'true',
    }
    if permit_q:
        params['permit_q'] = permit_q

    all_records = []
    page = 0

    while True:
        response = requests.get(BASE_URL, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()

        items = data.get('items', [])
        all_records.extend(items)
        page += 1

        if page == 1:
            total = data.get('total_count')
            print(f"Total records (API estimate): {total}")

        print(f"Page {page}: fetched {len(items)} records (running total: {len(all_records)})")

        next_cursor = data.get('next_cursor')
        if not next_cursor:
            break

        params['cursor'] = next_cursor
        params.pop('include_count', None)
        time.sleep(0.2)

    return all_records

In [23]:
# Try different heat pump keyword variants to find what's used in IL descriptions
hp_terms = ['heat pump', 'ASHP', 'air source', 'mini split', 'minisplit', 'mini-split', 'geothermal', 'ground source']

for term in hp_terms:
    r = requests.get(
        BASE_URL,
        headers=HEADERS,
        params={'geo_id': STATE, 'permit_from': DATE_FROM, 'permit_to': DATE_TO,
                'permit_tags': 'hvac', 'permit_q': term, 'property_type': 'residential',
                'size': 3, 'include_count': 'true'},
    )
    r.raise_for_status()
    data = r.json()
    total = data.get('total_count', {})
    count = total.get('value', 0) if isinstance(total, dict) else total
    # Show sample descriptions
    descs = [item.get('description', '')[:80] for item in data.get('items', [])]
    print(f"'{term}': {count} results")
    for d in descs:
        print(f"   {d}")
    time.sleep(0.3)

'heat pump': 11 results
   Replace gas furnace with heat pump heater
   Install mitsubishi 4 zone heat pump system. Includes 1 outdoor unit and 4 indoor
   Install heat pump
'ASHP': 0 results
'air source': 0 results
'mini split': 11 results
   Installation of air conditioning system - mini split conditioner
   Installation of new mini split a/c system with 17 seer condenser.
   3 season mini split
'minisplit': 2 results
   Ac ductless minisplit for bedroom, second floor
   Sfd - room addition - minisplit
'mini-split': 9 results
   Installation of a mitsubishi 15k mini-split with hyper heat. The mini-split unit
   Mini-split
   A/c in new location. Mchenry heating and air will be installing a mini-split ac 
'geothermal': 6 results
   Install geothermal
   Geothermal - htg and cooling
   Furnace and ac replacement - geothermal
'ground source': 1 results
   Install a closed loop geothermal ground source heat pump consisting of 3 vertica


In [24]:
# Fetch heat pump permits (hvac-tagged permits filtered to heat pump descriptions)
print('--- Fetching heat pump permits ---')
hp_records = fetch_all_permits(
    geo_id=STATE,
    permit_from=DATE_FROM,
    permit_to=DATE_TO,
    permit_tags='hvac',
    permit_q='heat pump',
)
hp_df = pd.json_normalize(hp_records)
hp_df['tag'] = 'heat_pump'
print(f"\nHeat pump records: {len(hp_df):,}")

--- Fetching heat pump permits ---
Total records (API estimate): {'value': 11, 'relation': 'eq'}
Page 1: fetched 11 records (running total: 11)

Heat pump records: 11


In [25]:
permits = hp_df.copy()

print(f"Total heat pump permits: {len(permits):,}")
print(f"Columns: {list(permits.columns)}")
permits.head()

Total heat pump permits: 11
Columns: ['property_census_tract', 'property_congressional_district', 'property_type', 'property_type_detail', 'property_legal_owner', 'property_owner_type', 'property_lot_size', 'property_building_area', 'property_story_count', 'property_unit_count', 'property_year_built', 'property_assess_market_value', 'id', 'number', 'description', 'jurisdiction', 'job_value', 'type', 'subtype', 'fees', 'status', 'file_date', 'issue_date', 'final_date', 'start_date', 'end_date', 'total_duration', 'construction_duration', 'approval_duration', 'inspection_pass_rate', 'contractor_id', 'tags', 'address.street_no', 'address.street', 'address.city', 'address.county', 'address.zip_code', 'address.zip_code_ext', 'address.state', 'address.jurisdiction', 'address.latlng', 'geo_ids.address_id', 'geo_ids.city_id', 'geo_ids.county_id', 'geo_ids.jurisdiction_id', 'tag']


,property_census_tract,property_congressional_district,property_type,property_type_detail,property_legal_owner,property_owner_type,property_lot_size,property_building_area,property_story_count,property_unit_count,property_year_built,property_assess_market_value,id,number,description,jurisdiction,job_value,type,subtype,fees,status,file_date,issue_date,final_date,start_date,end_date,total_duration,construction_duration,approval_duration,inspection_pass_rate,contractor_id,tags,address.street_no,address.street,address.city,address.county,address.zip_code,address.zip_code_ext,address.state,address.jurisdiction,address.latlng,geo_ids.address_id,geo_ids.city_id,geo_ids.county_id,geo_ids.jurisdiction_id,tag
0,171118708.082053,11,residential,Single Family,Norman Bair,individual,13504.0,1120.0,1.0,1.0,1953.0,17879700.0,c04d3065df291c83,BLD-2020-03209,Replace gas furnace with heat pump heater,CRYSTAL LAKE,1300000.0,Instant,Na,0,final,2020-11-16,2020-11-16,2021-03-24,2020-11-16,2021-03-24,128,128.0,0,None,None,"[heat_pump, hvac]",321,POPLAR ST,CRYSTAL LAKE,MCHENRY,60014,4410,IL,CRYSTAL LAKE,"[42.242289, -88.309583]",DYU4iiWso84,TRncIIqa2fY,zYwFGHeHcmA,jcQ4TVyH6yI,heat_pump
1,809100,09,residential,Single Family,MORGAN KELLY & TAMI YEAGER JOINT REVOCABLE TR;...,company_owned,4950.0,1052.0,1.0,1.0,1926.0,39805000.0,a1335bbd8017a6cf,20HVAC-0053,Install mitsubishi 4 zone heat pump system. In...,EVANSTON,0.0,Hvac only,1926 noyes,0,final,2020-08-23,2020-08-23,2021-08-06,2020-08-23,2021-08-06,348,348.0,0,None,7FDGCiuawN,"[electrical, hvac]",1926,NOYES ST,EVANSTON,COOK,60201,2555,IL,EVANSTON,"[42.057952, -87.700105]",DV0vDcTBBx8,TWuQfEGsQBw,zTsCn0ubu8s,jYm_nMtK3ck,heat_pump
2,809300,09,residential,Single Family,CAROLYN H KRULEE TRUST,company_owned,5142.0,2385.0,2.0,1.0,1925.0,71553000.0,40499ac68db891c7,20HVAC-0036,Install heat pump,EVANSTON,0.0,Hvac only,1305 grant street,0,final,2020-04-22,2020-04-22,2020-05-06,2020-04-22,2020-05-06,14,14.0,0,None,9OdlkZfPrG,"[heat_pump, hvac]",1305,GRANT ST,EVANSTON,COOK,60201,2624,IL,EVANSTON,"[42.059644, -87.689846]",DdhxlpwfloQ,TWuQfEGsQBw,zTsCn0ubu8s,jYm_nMtK3ck,heat_pump
3,000500,13,residential,Single Family,Graham K Bromley Revocable Tru; Graham K Bromley,company_owned,16401.0,1778.0,1.0,1.0,1953.0,27510000.0,d118f29dc350d02d,BS19-3044,Heat pump replacement,CHAMPAIGN,NaN,Hvac,Repair/replace,16900,final,2019-12-06,2019-12-06,2019-12-10,2019-12-06,2019-12-10,4,4.0,0,None,SsKfJjEYof,"[heat_pump, hvac]",701,HAMILTON DR,CHAMPAIGN,CHAMPAIGN,61820,6811,IL,CHAMPAIGN,"[40.100168, -88.253883]",DZ9oKoLCJL0,TccuBcC5Kls,zcwk8ik68C4,jcwk8ik68C4,heat_pump
4,171118712.023001,09,residential,Single Family,Jason A Seleski,individual,10845.0,1716.0,1.0,1.0,1972.0,27350100.0,79c6d1d3e553c09f,BLD-2019-01901,Install a closed loop geothermal ground source...,CRYSTAL LAKE,3000000.0,Residential alteration,Alteration,0,final,2019-07-29,2019-07-29,2020-05-29,2019-07-29,2020-05-29,305,305.0,0,None,Z5iACSzQei,[hvac],37,BERKSHIRE DR,CRYSTAL LAKE,MCHENRY,60014,3212,IL,CRYSTAL LAKE,"[42.219319, -88.321643]",DUR-l5-A1I0,TRncIIqa2fY,zYwFGHeHcmA,jcQ4TVyH6yI,heat_pump
